# Hisaab Aunty: complete Hinglish training (one notebook, press Run all)

**What it does**
1. Loads your model from Google Drive.
2. Gets our generated Hinglish training data from GitHub (about 5,300 labelled examples).
3. Downloads public datasets from Hugging Face and uses them as **negatives**: everyday Hinglish chat and bank-support questions that must NOT trigger any action. They are labelled `unknown`.
4. Trains a LoRA adapter, merges it, saves a full model to Drive.
5. Tests it on held-out messages.

**Honest limits**
- Public datasets do not contain Aunty's intents (log a spend, status, undo...). Those examples are ours (synthetic). The public data teaches the model to stay quiet on unrelated chat.
- This notebook has never been run by its author (no GPU on their laptop). If a cell fails, copy the last 10 lines of the error to Claude.
- Check each dataset's license on its Hugging Face page before using the trained model commercially. Sources are written to DATA_SOURCES.md next to the trained model.

**Before you press Run all**
- Runtime menu: Change runtime type, pick T4 GPU, Save.
- Google Drive must contain the folder `Sent_model` with: config.json, tokenizer.json, tokenizer_config.json, chat_template.jinja, generation_config.json, model.safetensors.

In [ ]:
# 1. GPU, Drive, settings
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, re, shutil, itertools

CANDIDATES = ['/content/drive/MyDrive/Sent_model', '/content/drive/MyDrive/Hisaab/Sent_model']   # the notebook uses whichever exists
BASE_DRIVE = next((p for p in CANDIDATES if os.path.exists(p + '/model.safetensors')), None)
assert BASE_DRIVE, 'Model folder not found. Put your model folder in Google Drive as My Drive/Sent_model'
OUT_DRIVE = os.path.dirname(BASE_DRIVE) + '/Sent_model_hinglish'   # the trained model is saved next to it
EPOCHS = 2                   # 1 = faster first try
N_HINGLISH_NEG = 1500        # everyday Hinglish lines used as "do nothing" examples
N_BANK_NEG = 1000            # English bank-support questions used as "do nothing" examples
N_TEST_REAL = 80             # held-out real negatives for the final check

# every file the trainer needs must be there; tokenizer.json is the one people forget
need = ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
missing = [f for f in need if not os.path.exists(BASE_DRIVE + '/' + f)]
print('model folder:', BASE_DRIVE)
assert not missing, 'MISSING in the Drive folder ' + BASE_DRIVE + ': ' + ', '.join(missing) + '. Upload it from your PC folder Documents/Sent_model.'
print('all model files found')

In [ ]:
# 2. Install and get our training data from GitHub
!pip install -q -U transformers peft trl datasets accelerate
!rm -rf /content/repo && git clone -q https://github.com/sohail807/hisaab_aunty /content/repo
!ls -la /content/repo/prototype/training/data

In [ ]:
# 3. Copy the model to the fast local disk (skips the big .gguf files)
os.makedirs('/content/base', exist_ok=True)
for f in os.listdir(BASE_DRIVE):
    p = os.path.join(BASE_DRIVE, f)
    if os.path.isfile(p) and not f.endswith('.gguf'):
        shutil.copy2(p, '/content/base/' + f)
print(sorted(os.listdir('/content/base')))

In [ ]:
# 4. Load our labelled data
random.seed(7)
R = '/content/repo/prototype/training/data'
def read(p): return [json.loads(l) for l in open(p, encoding='utf-8')]
train, val, test = read(R + '/train.jsonl'), read(R + '/val.jsonl'), read(R + '/test.jsonl')
systems = [r['messages'][0]['content'] for r in train]   # the exact instruction text the app sends
print('our data: train', len(train), 'val', len(val), 'test', len(test))

In [ ]:
# 5. Public datasets used as NEGATIVES (messages that must produce {"intent":"unknown"})
# Anything about money, banking actions, greetings or tax is filtered OUT, so no negative can contradict our real intents.
from datasets import load_dataset
SKIP = re.compile(r"(\d|₹|\brs\b|rupee|rupay|paisa|paise|kharch|bacha|budget|hisaab|bill|udhaar|salary|kamai|bank|loan|emi|khata|price|sasta|mehenga|amount|balance|account|limit|spend|spent|expense|save|saving|goal|streak|recap|undo|delete|cancel|transfer|payment|paid|\bpay\b|card|cash|upi|refund|charge|fee|tax|invest|share|stock|crypto|bitcoin|fund|insurance|\bhi\b|hello|namaste|kaise|kaisi|kya haal|thank|shukriya|dhanyavaad|bye|good (morning|night)|aunty|http|www|@)", re.I)

def strings(x):
    if isinstance(x, str): yield x
    elif isinstance(x, dict):
        for v in x.values(): yield from strings(v)
    elif isinstance(x, (list, tuple)):
        for v in x: yield from strings(v)

def clean_lines(rows, limit_rows):
    out = set()
    for i, row in enumerate(rows):
        if i >= limit_rows: break
        for s in strings(row):
            for ln in s.split('\n'):
                ln = re.sub(r'^\s*(person\s*\d*|user|speaker\s*\d*|human|assistant|[AB])\s*[:\-]\s*', '', ln, flags=re.I).strip()
                w = ln.split()
                if 3 <= len(w) <= 14 and re.fullmatch(r"[A-Za-z ,.'?!\-]+", ln) and not SKIP.search(ln):
                    out.add(ln)
    return sorted(out)

sources = []
hinglish, bank = [], []
try:
    ds = load_dataset('Abhishekcr448/Hinglish-Everyday-Conversations-1M', split='train', streaming=True)
    hinglish = clean_lines(ds, 30000)
    sources.append('Abhishekcr448/Hinglish-Everyday-Conversations-1M (license reported: MIT; synthetic, made with GPT-4o-mini)')
except Exception as e:
    print('SKIPPED Hinglish chat dataset:', repr(e)[:300])
for name in ['PolyAI/banking77', 'legacy-datasets/banking77']:
    try:
        ds = load_dataset(name, split='train')
        bank = clean_lines(({'t': r['text']} for r in ds), 100000)
        sources.append(name + ' (Banking77, license reported: CC BY 4.0)')
        break
    except Exception as e:
        print('could not load', name, repr(e)[:200])
print('usable Hinglish lines:', len(hinglish), '| usable bank lines:', len(bank))
print('examples:', random.sample(hinglish, min(5, len(hinglish))), random.sample(bank, min(3, len(bank))))

In [ ]:
# 6. Build the mixed training files
def neg(text): return {'id': 'real_neg', 'intent': 'unknown', 'messages': [{'role': 'system', 'content': random.choice(systems)}, {'role': 'user', 'content': text}, {'role': 'assistant', 'content': '{"intent":"unknown"}'}]}
random.shuffle(hinglish); random.shuffle(bank)
held = hinglish[:N_TEST_REAL // 2] + bank[:N_TEST_REAL // 2]                       # never trained on
hinglish, bank = hinglish[N_TEST_REAL // 2:], bank[N_TEST_REAL // 2:]
real_train = [neg(t) for t in hinglish[:N_HINGLISH_NEG]] + [neg(t) for t in bank[:N_BANK_NEG]]
real_val = [neg(t) for t in hinglish[N_HINGLISH_NEG:N_HINGLISH_NEG + 60]] + [neg(t) for t in bank[N_BANK_NEG:N_BANK_NEG + 40]]
mix_train, mix_val = train + real_train, val + real_val
random.shuffle(mix_train); random.shuffle(mix_val)
os.makedirs('/content/mix', exist_ok=True)
for name, rows in (('train', mix_train), ('val', mix_val), ('test_real', [neg(t) for t in held])):
    open('/content/mix/' + name + '.jsonl', 'w', encoding='utf-8').write('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + '\n')
print('train', len(mix_train), '(of which real negatives', len(real_train), ') | val', len(mix_val), '| held-out real negatives', len(held))

## 7. Train
Runs in its own process so GPU memory is fully freed afterwards. Keep this browser tab open. A free T4 can take a few hours for this much data (not measured). If it disconnects, lower EPOCHS to 1 in cell 1 and run again.

In [ ]:
!python /content/repo/prototype/training/train_lora.py --base /content/base --data /content/mix --out /content/out --epochs $EPOCHS

In [ ]:
# 8. Save the trained model and its data credits to Drive
shutil.copytree('/content/out', OUT_DRIVE, dirs_exist_ok=True)
open(OUT_DRIVE + '/DATA_SOURCES.md', 'w').write('# Data used to train this model\n\n- Our own generated Hinglish intent data: https://github.com/sohail807/hisaab_aunty (prototype/training/gen_dataset.js)\n' + ''.join('- ' + s + '\n' for s in sources) + '\nCheck each dataset license before commercial use. Share-alike or attribution terms may apply.\n')
print('saved to', OUT_DRIVE)

In [ ]:
# 9. Test the new model on held-out messages (greedy answers, same format as the app)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained('/content/out')
model = AutoModelForCausalLM.from_pretrained('/content/out', torch_dtype=torch.float16, device_map='auto')
step = max(1, len(test) // 60)
sample = [r for i, r in enumerate(test) if i % step == 0][:60] + [json.loads(l) for l in open('/content/mix/test_real.jsonl')]
def ask(r):
    prompt = tok.apply_chat_template(r['messages'][:2], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=120, do_sample=False)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
def intent_of(s):
    m = re.search(r'\{.*\}', s, re.S)
    try: return json.loads(m.group(0)).get('intent') if m else None
    except Exception: return None
ok = valid = wrong_saves = n = real_n = real_ok = 0; bad = []
for r in sample:
    got = ask(r); gold = json.loads(r['messages'][2]['content'])['intent']; pred = intent_of(got); n += 1
    valid += pred is not None; ok += pred == gold
    if pred == 'log_spend' and gold != 'log_spend': wrong_saves += 1
    if r.get('id') == 'real_neg':
        real_n += 1; real_ok += pred == 'unknown'
    if pred != gold and len(bad) < 12: bad.append((r['messages'][1]['content'], gold, pred))
print(f'messages {n} | valid JSON {valid}/{n} | right intent {ok}/{n} ({100*ok//n}%)  [baseline before training: 59%]')
print(f'WRONG SAVES (model said log_spend but should not): {wrong_saves}   [must be 0]')
print(f'real unrelated chat correctly left alone: {real_ok}/{real_n}')
for b in bad: print('MISS', b)

## Done
The trained model is in Drive: `Hisaab/Sent_model_hinglish` (about 6 GB), with DATA_SOURCES.md.
Next on your PC: download it, convert to GGUF, add it to `models.json`, run `node training/eval_model.js --model <id> --n 70`, and keep it only if it beats the baseline (59% right intent, 80% right amounts, 2 wrong saves) with 0 wrong saves.